# einops-reduce composite — cx10: row mean then pair every row-mean with every column via repeat-broadcast

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-reduce`, `einops-repeat-broadcast`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-reduce"
DD_ATOM_IDS = ["einops-reduce", "einops-repeat-broadcast"]
DD_SUBTOPICS = ["Einops: Reduce", "Einops: Repeat-as-broadcast"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`einops.repeat` has two distinct uses: introducing a new named axis (kwarg-bound), and broadcasting an existing tensor against another by inserting a zero-stride axis. ARENA's ray-tracing pair-every-with-every drill is the canonical repeat-as-broadcast example.

Here we compose it with reduce. First reduce a 2-D `(R, C)` matrix to its per-row mean of shape `(R,)`. Then use `repeat` to broadcast the row-means against every column index, yielding a `(R, N)` 'pair-every-with-every' tensor where row r, column n holds the r-th row mean. This is repeat-as-broadcast: no copy semantics — einops materialises a zero-stride view.

### Composite Exercise — row mean then pair every row-mean with every column via repeat-broadcast

**Atoms exercised together**: `einops-reduce`, `einops-repeat-broadcast`

Implement `cx10_row_mean_paired_with_n_cols(x, n)` that takes a `(R, C)` matrix and an int `n` and returns a `(R, n)` tensor where every column holds the per-row mean of `x`.

Two atoms in one fn:

1. **Reduce** — collapse the column axis with `einops.reduce(x, 'r c -> r', 'mean')`.
2. **Repeat-as-broadcast** — use `einops.repeat(row_means, 'r -> r n', n=n)` to pair every row-mean with every output column. This is the same pattern as ARENA's ray-with-triangle pairing: insert a new axis whose size is bound by kwarg, and every slot along the new axis holds the same row-mean (zero-stride broadcast semantics).

Sanity check inside the fn: `assert row_means.shape == (R,)` so atom 1 is visible.

In [ ]:
def cx10_row_mean_paired_with_n_cols(x, n):
    # Atom 1: reduce — drop the C axis to get per-row scalars.
    row_means = reduce(x, 'r c -> r', 'mean')
    assert row_means.shape == (x.shape[0],), row_means.shape
    # Atom 2: repeat-as-broadcast — insert a new 'n' axis bound by kwarg; every slot is
    # the same row_mean. einops models this as a zero-stride broadcast under the hood.
    return repeat(row_means, 'r -> r n', n=n)


<details><summary>Show solution — cx10</summary>

```python
def cx10_row_mean_paired_with_n_cols(x, n):
    # Atom 1: reduce — drop the C axis to get per-row scalars.
    row_means = reduce(x, 'r c -> r', 'mean')
    assert row_means.shape == (x.shape[0],), row_means.shape
    # Atom 2: repeat-as-broadcast — insert a new 'n' axis bound by kwarg; every slot is
    # the same row_mean. einops models this as a zero-stride broadcast under the hood.
    return repeat(row_means, 'r -> r n', n=n)
```

Repeat is not always 'copy data' — when used to pair every X with every Y, it's a broadcast/expand under the hood. einops's contract is on shape, not memory: the output looks like every column holds the row-mean, and the framework picks the most efficient impl.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx10',
        'subtopics': ["Einops: Reduce", "Einops: Repeat-as-broadcast"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()